# Stage 2 - Blocking (Candidate Generation)

This notebook contains:
1. **Environment & Paths**: Verify working directories, imports and the blocking configuration
2. **Design Evidence**: Training-data facts the blocking design relies on (country partition, matches per source, Indic scripts)
3. **Blocking Normalization**: Transliteration, legal/filler tokens, leet digits and phonetic keys on real records
4. **Blocking Keys**: The feature views each retrieval pass indexes, shown on a real matched pair
5. **Validation Run**: Val-split S1 queries vs the FULL train S2/S3 (~10.3M records), scored against ground truth
6. **Budget Trade-off**: Recall / F0.5 ceiling vs candidates kept per source
7. **Candidate Inspection**: Ranked candidates for sample queries, true matches marked
8. **Rescoring Weights (optional)**: Refit the logistic-regression ranking weights
9. **Training Candidates**: Scored candidate pairs for a 200k train sample (Stage 3 training data)
10. **Test Candidates**: `output/candidate_pairs.tsv` for all test S1 entities + submission-rule validation

Prerequisites (notebook `01_preprocessing.ipynb`): Stage 1 output in `dataset/preprocessed/` and the validation split in `dataset/val_split/`.
Heavy steps (val ~9 min, train ~14 min, test ~40 min on 22 threads) are skipped when their outputs already exist; set `RERUN = True` to regenerate.

In [ ]:
# 1. Environment & Path Setup
import os
import sys
import json
import random
import pandas as pd
import numpy as np

# Resolve project root dynamically (works whether opened from root or notebooks/)
CWD = os.path.abspath(os.getcwd())
ROOT_DIR = CWD if os.path.exists(os.path.join(CWD, 'src')) else os.path.abspath(os.path.join(CWD, '..'))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from src.blocking import BlockingConfig, evaluate_candidates, format_report, load_source

DATA_DIR = os.path.join(ROOT_DIR, 'student_resource', 'dataset')
PRE_DIR = os.path.join(ROOT_DIR, 'dataset', 'preprocessed')
VAL_GT = os.path.join(ROOT_DIR, 'dataset', 'val_split', 'val_ground_truth.tsv')
CAND_DIR = os.path.join(ROOT_DIR, 'dataset', 'candidates')
OUT_DIR = os.path.join(ROOT_DIR, 'output')
SCRIPTS_DIR = os.path.join(ROOT_DIR, 'scripts')
RERUN = False  # True = regenerate val/train/test candidates even if outputs exist

os.makedirs(CAND_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Python Version : {sys.version.split()[0]}")
print(f"Pandas Version : {pd.__version__}")
print(f"Root Directory : {ROOT_DIR}")
print(f"Preprocessed   : {sorted(os.listdir(PRE_DIR)) if os.path.exists(PRE_DIR) else 'Not Found'}")
print(f"Val GT present : {os.path.exists(VAL_GT)}")

cfg = BlockingConfig()
print("\nRetrieval passes:")
for p in cfg.passes:
    print(f"  {p.name:<10} k={p.k:<3} max_df={p.max_df:<6} views={p.views}")
print(f"Rescoring weights : {cfg.rescore_weights}")
print(f"Budget            : {cfg.max_candidates_per_source} candidates per source (S2, S3)")


### 2. Design Evidence from the Training Data
- **Country partition is lossless**: if no true pair crosses country labels, S1 queries only need to be compared with S2/S3 records of the same country. The label is treated as an open set (France appears only in test).
- **Matches per source** bound the useful candidate budget.
- **Indic scripts**: S1 is ASCII, but S2/S3 India names are often in Devanagari, Tamil, Telugu, ... (handled by Stage 1 transliteration).

In [ ]:
# 2. Country consistency and match counts over the full ground truth
gt = pd.read_csv(os.path.join(DATA_DIR, "train", "train_ground_truth.tsv"), sep="\t", dtype=str, keep_default_na=False)
pairs = gt.assign(candidate=gt["matched_entity_ids"].str.split(",")).explode("candidate")
pairs = pairs[pairs["candidate"] != ""]

country = pd.concat([
    load_source(PRE_DIR, "train", n, columns=["entity_id", "country"]) for n in (1, 2, 3)
]).set_index("entity_id")["country"]
cross = (pairs["source1_entity_id"].map(country) != pairs["candidate"].map(country)).sum()

per_source = pairs.assign(source=pairs["candidate"].str[:2]).groupby(["source1_entity_id", "source"]).size()
print(f"True pairs                     : {len(pairs):,}")
print(f"Pairs crossing country labels  : {cross:,}")
print(f"Singleton S1 entities          : {(gt['matched_entity_ids'] == '').mean():.2%}")
print(f"Matches per S1 (mean)          : {len(pairs) / len(gt):.2f}")
print("Max matches per S1 per source  :", per_source.groupby(level="source").max().to_dict())
del country


In [ ]:
# 3. Transliteration of Indic text
from src.preprocessing.text import transliterate_text

def indic_share(tsv_path):
    names = pd.read_csv(tsv_path, sep="\t", dtype=str, keep_default_na=False, nrows=50_000)["name"]
    return f"{(names.map(transliterate_text) != names).mean():.1%}"

print("Fraction of records with non-Latin text requiring transliteration (first 50k):")
for name in ("train_source1", "train_source2", "train_source3"):
    file_path = os.path.join(DATA_DIR, "train", f"{name}.tsv")
    print(f"{name}: {indic_share(file_path)}")


### 3. Blocking Normalization on Real Records
Stage 1 (`clean_text`) transliterates Indic scripts before de-accenting. On top of it, blocking (`src/blocking/normalize.py`):
- drops legal forms / filler anywhere in the name (`Ltd. Ontime Pvt. Services`, `M/s`, `D.B.A.`),
- maps digits inside words back to letters (`C0astal`, `5ervices`),
- builds a phonetic skeleton that survives transliteration (`kulopal pichinas` ~ `global business`).

In [ ]:
# Normalisation transforms on real records from train_source2
from src.preprocessing.text import (
    legal_clean, filler_clean, number_words_to_digits, normalize_whitespace,
)
s2_path = os.path.join(DATA_DIR, "train", "train_source2.tsv")
s2_head = pd.read_csv(s2_path, sep="\t", dtype=str, keep_default_na=False, nrows=200_000)
sample = s2_head[s2_head["name"].str.contains("pvt\\. ltd\\.|inc\\.|llc|gmbh", case=False, regex=True)].sample(5, random_state=7)

print(f"{'RAW':<40} {'LEGAL REMOVED':<30} {'LEET / NUMBER WORDS'}")
print("-" * 95)
for raw in sample["name"]:
    no_legal = legal_clean(raw)
    num_norm = number_words_to_digits(no_legal)
    print(f"{raw[:38]:<40} {no_legal[:28]:<30} {num_norm[:28]}")


### 4. Blocking Keys (Feature Views)
Single name words are common because the corpus reuses a small business vocabulary. Word **pairs**, address **bigrams** and **name x address** cross keys are rare and identifying. Retrieval indexes only keys with document frequency <= `max_df` (1,000), which bounds the cost per query.

In [ ]:
# 4. Keys of a real matched pair from the validation split, and the keys they share
from src.blocking.features import record_features

val_gt = pd.read_csv(VAL_GT, sep="\t", dtype=str, keep_default_na=False)
val_pairs = val_gt.assign(candidate=val_gt["matched_entity_ids"].str.split(",")).explode("candidate")
val_pairs = val_pairs[val_pairs["candidate"] != ""]
s1_id, t_id = val_pairs.sample(1, random_state=random.randint(0, 10_000)).iloc[0][["source1_entity_id", "candidate"]]

def lookup(entity_id):
    src_num = int(entity_id[1])
    parquet_path = os.path.join(PRE_DIR, f"train_source{src_num}.parquet")
    tsv_path = os.path.join(PRE_DIR, f"train_source{src_num}.tsv")
    if os.path.exists(parquet_path):
        return pd.read_parquet(parquet_path, filters=[("entity_id", "==", entity_id)]).iloc[0]
    else:
        df = pd.read_csv(tsv_path, sep="\t", dtype=str, keep_default_na=False)
        return df[df["entity_id"] == entity_id].iloc[0]

a, b = lookup(s1_id), lookup(t_id)
print(f"S1: {a.clean_name} | {a.clean_address}")
print(f"{t_id[:2]}: {b.clean_name} | {b.clean_address}\n")
for view in dict.fromkeys(v for p in cfg.passes for v in p.views):
    fa, fb = record_features(a.clean_name, a.clean_address, view), record_features(b.clean_name, b.clean_address, view)
    print(f"{view:<15} shared={sorted(set(fa) & set(fb))[:6]}  (S1 has {len(fa)} keys)")


### 5. Validation Run
The 10k validation S1 queries are searched against the **full** train S2/S3, so candidate-set sizes and recall reflect the real 1.7M x 10M problem (a small distractor pool would make blocking look better than it is).

In [ ]:
# 5. Blocking on the validation split (~9 min); skipped if the report already exists
val_report_path = os.path.join(CAND_DIR, "val_blocking_report.json")
run_blocking_script = os.path.join(SCRIPTS_DIR, "run_blocking.py")
if RERUN or not os.path.exists(val_report_path):
    !python "{run_blocking_script}" --split val --preprocessed-dir "{PRE_DIR}" --val-gt "{VAL_GT}" --out-dir "{CAND_DIR}"
else:
    print(f"Using existing {val_report_path} (set RERUN = True to regenerate)")

val_report = json.load(open(val_report_path))
print(f"\nRun time: {val_report['seconds'] / 60:.1f} min | step seconds: {val_report['step_seconds']}\n")
print(format_report(val_report["metrics"]))


In [ ]:
# 6. Budget trade-off: recall and F0.5 ceiling if fewer candidates per source are kept
val_pairs_path = os.path.join(CAND_DIR, "val_candidate_pairs.parquet")
val_pairs_scored = pd.read_parquet(val_pairs_path)
rows = []
for budget in (1, 3, 5, 10, 15, 20, 25):
    r = evaluate_candidates(val_pairs_scored[val_pairs_scored["source_rank"] < budget], val_gt)
    rows.append({"per_source": budget, "per_S1": 2 * budget, "pair_recall": r["pair_recall"],
                 "f05_ceiling": r["f05_ceiling"], "zero_recall_queries": r["query_zero_recall"]})
pd.DataFrame(rows).round(4)


In [ ]:
# 7. Ranked candidates for a random validation query (true matches marked)
qid = random.choice(val_gt.loc[val_gt["matched_entity_ids"] != "", "source1_entity_id"].tolist())
truth = set(val_gt.set_index("source1_entity_id").loc[qid, "matched_entity_ids"].split(","))
cands = val_pairs_scored[val_pairs_scored["source1_entity_id"] == qid].sort_values("candidate_score", ascending=False)

ids = cands["candidate_entity_id"].tolist()
dfs = []
for n in (2, 3):
    p_path = os.path.join(PRE_DIR, f"train_source{n}.parquet")
    t_path = os.path.join(PRE_DIR, f"train_source{n}.tsv")
    if os.path.exists(p_path):
        dfs.append(pd.read_parquet(p_path, filters=[("entity_id", "in", ids)]))
    else:
        d = pd.read_csv(t_path, sep="\t", dtype=str, keep_default_na=False, usecols=["entity_id", "clean_name", "clean_address", "country"])
        dfs.append(d[d["entity_id"].isin(ids)])
records = pd.concat(dfs).set_index("entity_id")

q = lookup(qid)
print(f"Query {qid}: {q.clean_name} | {q.clean_address}  ({len(truth)} true matches)")
view = cands.assign(
    true=cands["candidate_entity_id"].isin(truth),
    name=cands["candidate_entity_id"].map(records["clean_name"]),
    address=cands["candidate_entity_id"].map(records["clean_address"]),
)[["candidate_entity_id", "true", "source_rank", "candidate_score", "name", "address"]]
view.head(12)


### 8. Rescoring Weights (optional)
The retrieved union is ranked by a weighted sum of exact similarities. The weights in `src/blocking/config.py` are logistic-regression coefficients (true pair vs not) fitted on validation candidates. This cell refits them and reports held-out recall for current vs fitted weights; the result is saved as a config usable with `run_blocking.py --config`.

In [ ]:
# 8. Refit rescoring weights (~10 min); off by default
REFIT = False
fitted_path = os.path.join(CAND_DIR, "fitted_blocking_config.json")
fit_script = os.path.join(SCRIPTS_DIR, "fit_rescore_weights.py")
if REFIT:
    !python "{fit_script}" --preprocessed-dir "{PRE_DIR}" --gt "{VAL_GT}" --out "{fitted_path}"
if os.path.exists(fitted_path):
    print("Fitted weights :", json.load(open(fitted_path))["rescore_weights"])
print("Default weights:", cfg.rescore_weights)


### 9. Training Candidates for Stage 3
A random 200k train S1 sample (validation entities excluded) is blocked against the full train S2/S3. The scored pairs (`score_*` retrieval cosines, `sim_*` exact similarities, `candidate_score`, `source_rank`) are the Stage 3 training data. Because the rescoring weights were fitted on the validation split, these metrics are also the unbiased estimate of blocking quality.

In [ ]:
# 9. Training candidates (~14 min); skipped if the report already exists
train_report_path = os.path.join(CAND_DIR, "train_blocking_report.json")
run_blocking_script = os.path.join(SCRIPTS_DIR, "run_blocking.py")
train_gt_path = os.path.join(DATA_DIR, "train", "train_ground_truth.tsv")
if RERUN or not os.path.exists(train_report_path):
    !python "{run_blocking_script}" --split train --sample-s1 200000 --preprocessed-dir "{PRE_DIR}" --val-gt "{VAL_GT}" --train-gt "{train_gt_path}" --out-dir "{CAND_DIR}"
else:
    print(f"Using existing {train_report_path} (set RERUN = True to regenerate)")

print(format_report(json.load(open(train_report_path))["metrics"]))
train_head = pd.read_parquet(os.path.join(CAND_DIR, "train_candidate_pairs.parquet")).head()
train_head.astype({c: "float32" for c in train_head.columns if str(train_head[c].dtype) == "float16"})


### 10. Test Candidates
All 1.73M test S1 entities (US, India and the unseen France) are blocked against the test S2/S3. The output `output/candidate_pairs.tsv` is checked against the submission rules of `student_resource/utils/validate_submission.py`, including the check that every candidate ID exists in the test S2/S3 files.

In [ ]:
# 10. Test candidates (~40 min); skipped if output/candidate_pairs.tsv already exists
test_tsv = os.path.join(OUT_DIR, "candidate_pairs.tsv")
run_blocking_script = os.path.join(SCRIPTS_DIR, "run_blocking.py")
if RERUN or not os.path.exists(test_tsv):
    !python "{run_blocking_script}" --split test --preprocessed-dir "{PRE_DIR}" --out-dir "{CAND_DIR}" --submission-dir "{OUT_DIR}"
else:
    print(f"Using existing {test_tsv} (set RERUN = True to regenerate)")


In [ ]:
# Validate candidate_pairs.tsv with the official validator's rules (matching_results.tsv comes in Stage 5)
import importlib.util

val_script = os.path.join(ROOT_DIR, "student_resource", "utils", "validate_submission.py")
spec = importlib.util.spec_from_file_location("validate_submission", val_script)
validator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validator)

test_dir = os.path.join(DATA_DIR, "test")
required = validator.read_ids(os.path.join(test_dir, "test_source1.tsv"))
warnings, errors = [], []
valid_ids = validator.load_match_targets(test_dir, warnings)
mapping = validator.validate_id_list_file(test_tsv, validator.CANDIDATE_HEADER, "candidate_entity_ids",
                                          required, valid_ids, errors)
sizes = pd.Series({k: len(v) for k, v in mapping.items()})
print(f"Candidates per S1: mean {sizes.mean():.1f}, min {sizes.min()}, max {sizes.max()}")
print("PASS" if not errors else f"FAIL: {errors}", "| warnings:", warnings or "none")
del mapping, valid_ids
